In [25]:
# !pip install datasets

In [ ]:
from datasets import load_dataset
import re
from collections import Counter
from collections import defaultdict
import random
random.seed(42)

## Evaluation Dataset Inspection

In [ ]:
# GSM8K test set
gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")

# HumanEval test set
humaneval = load_dataset("openai/openai_humaneval", split="test")

# IFEval (only has 'train' split — this IS the eval set)
ifeval = load_dataset("google/IFEval", split="train")

GSM8K columns: ['question', 'answer']
First example:
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}

HumanEval columns: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']
First example:
{'task_id': 'HumanEval/0', 'prompt': 'from typing import List\n\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    """ Check if in given list of numbers, are any two numbers closer to each other than\n    given threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)\n    True\n    """\n', 'ca

In [ ]:
print("GSM8K test:", len(gsm8k_test))
print("HumanEval test:", len(humaneval))
print("IFEval:", len(ifeval))

GSM8K test: 1319
HumanEval test: 164
IFEval: 541


In [ ]:
# Pretty print without truncation
import json

print("\n=== GSM8K example ===")
print("Q:", gsm8k_test[0]["question"])
print("A:", gsm8k_test[0]["answer"])  # contains #### delimiter

print("\n=== HumanEval example ===")
print("prompt:", humaneval[0]["prompt"])
print("canonical_solution:", humaneval[0]["canonical_solution"])
print("test:", humaneval[0]["test"][:200])  # unit test code

print("\n=== IFEval example ===")
ex = ifeval[0]
print("prompt:", ex["prompt"])
print("instruction_id_list:", ex["instruction_id_list"])
print("kwargs:", ex["kwargs"])


=== GSM8K example ===
Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
A: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

=== HumanEval example ===
prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """

canonical_solution:     for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - el

In [41]:
print("\n=== HumanEval example ===")
print("prompt:", humaneval[2]["prompt"])
print("canonical_solution:", humaneval[2]["canonical_solution"])
print("test:", humaneval[2]["test"][:200])  # unit test code


=== HumanEval example ===
prompt: 

def truncate_number(number: float) -> float:
    """ Given a positive floating point number, it can be decomposed into
    and integer part (largest integer smaller than given number) and decimals
    (leftover part always smaller than 1).

    Return the decimal part of the number.
    >>> truncate_number(3.5)
    0.5
    """

canonical_solution:     return number % 1.0

test: 

METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate(3.5) == 0.5
    assert abs(candidate(1.33) - 0.33) < 1e-6
    assert abs(candidate(123.456) - 0.


## Training Data Inspection

In [3]:
# ── GSM8K train ──────────────────────────────────────────────
gsm8k_train = load_dataset("openai/gsm8k", "main", split="train")
print("=== GSM8K Train ===")
print("Size:", len(gsm8k_train))
print("Columns:", gsm8k_train.column_names)
print("Example:")
print("  question:", gsm8k_train[0]["question"])
print("  answer:", gsm8k_train[0]["answer"])

=== GSM8K Train ===
Size: 7473
Columns: ['question', 'answer']
Example:
  question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
  answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [4]:
# ── MetaMathQA ───────────────────────────────────────────────
math_train = load_dataset("meta-math/MetaMathQA", split="train")
print("=== MetaMathQA ===")
print("Size:", len(math_train))
print("Columns:", math_train.column_names)
print("Example:")
print("  query:",math_train[0]['query'])
print("  answer:", math_train[0]['response'])

=== MetaMathQA ===
Size: 395000
Columns: ['type', 'query', 'original_question', 'response']
Example:
  query: Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?
  answer: The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the formula $\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.
In this case, Joe's point is $(1,2)$ and Gracie's point is $(-1,1)$.
So the distance between their points is $\sqrt{((-1)-(1))^2+((1)-(2))^2}=\sqrt{(-2)^2+(-1)^2}=\sqrt{4+1}=\sqrt{5}$.
Therefore, Gracie and Joe's points are $\boxed{\sqrt{5}}$ units apart.
The answer is: \sqrt{5}


In [5]:
# ── Tulu-3 — stream it, do NOT load fully (939K rows) ────────

if_train = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
print("=== Tulu-3 SFT Mixture ===")

# Streaming datasets have no len() — get size from info instead
info = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
try:
    from datasets import load_dataset_builder
    builder = load_dataset_builder("allenai/tulu-3-sft-mixture")
    print("Size:", builder.info.splits["train"].num_examples)
except:
    print("Size: ~939,000 (from dataset card)")

# Inspect first few examples without downloading everything
print("Columns:", if_train.column_names)
examples = list(if_train.take(3))
for i, ex in enumerate(examples):
    print(f"\nExample {i}:")
    print("  source:", ex.get("source", "N/A"))
    print("  keys:", list(ex.keys()))
    msgs = ex.get("messages", [])
    for m in msgs:
        print(f"  [{m['role']}]:", str(m['content'])[:200])

=== Tulu-3 SFT Mixture ===
Size: 939343
Columns: ['id', 'messages', 'source']

Example 0:
  source: ai2-adapt-dev/oasst1_converted
  keys: ['id', 'messages', 'source']
  [user]: Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.
  [assistant]: Sure, here's an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:
``` 
# Configure the AWS provider
provider "

Example 1:
  source: ai2-adapt-dev/oasst1_converted
  keys: ['id', 'messages', 'source']
  [user]: ¿Por qué crees que cada año es más difícil tener una casa propia en comparación a décadas anteriores?
  [assistant]: Existen varios factores que pueden contribuir a que cada año sea más difícil tener una casa propia en comparación con décadas anteriores. Algunos de los factores más importantes son:
- El aumento del 

Example 2:
  source: ai2-adapt-dev/oasst1_converted


In [24]:
# ── OpenCodeInstruct — stream it (5M rows) ───────────────────
code_train = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)
print("=== OpenCodeInstruct ===")

try:
    from datasets import load_dataset_builder
    builder = load_dataset_builder("nvidia/OpenCodeInstruct")
    print("Size:", builder.info.splits["train"].num_examples)
except:
    print("Size: ~5M (from dataset card)")

print("Columns:", code_train.column_names)
examples = list(code_train.take(3))
for i, ex in enumerate(examples):
    print(f"\nExample {i}:")
    print("  keys:", list(ex.keys()))
    # Print first 200 chars of each field
    for k, v in ex.items():
        print(f"  {k}:", str(v)[:200])

=== OpenCodeInstruct ===
Size: 5000000
Columns: ['id', 'input', 'output', 'domain', 'generation_algorithm', 'llm_judgement', 'unit_tests', 'tests_execution_status', 'average_test_score']

Example 0:
  keys: ['id', 'input', 'output', 'domain', 'generation_algorithm', 'llm_judgement', 'unit_tests', 'tests_execution_status', 'average_test_score']
  id: 4d61a897c0f86467f5c504a62a5f7282
  input: You are given a list of `n` tasks, each represented as a tuple `(start, end)`, indicating the start and end times of the task. The tasks are sorted by their start times. Your goal is to determine the 
  output: ```python
def max_non_overlapping_tasks(tasks):
    """
    Returns the maximum number of non-overlapping tasks that can be selected from a list of tasks.
    
    :param tasks: List of tuples, where 
  domain: generic
  generation_algorithm: self-instruct
  llm_judgement: {"requirement_conformance": {"score": 5, "justification": "The solution fully meets the requirements by correctly impleme

In [17]:
print("GSM8K train:", len(gsm8k_train))
print("metamath train:", len(math_train))
print("IF train:", len(if_train))
print("code train:", len(code_train))

GSM8K train: 7473
metamath train: 1
IF train: 1
code train: 1


Check if fencing has to be stripped:

- grep -A 30 "def verify" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/humaneval/humaneval.py
- grep -A 20 "def find_code" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/humaneval/humaneval.py

Get everything - constants and record_to_sample functions

- cat /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/gsm8k/*.py

- grep -A 20 "INSTRUCTION" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/humaneval/humaneval.py | head -40

- grep -A 20 "def record_to_sample" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/humaneval/humaneval.py

- grep -A 20 "def record_to_sample" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/ifeval/ifeval.py

And the DATASET_PATH for both

- grep "DATASET_PATH" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/humaneval/humaneval.py
- grep "DATASET_PATH" /opt/miniconda3/envs/cps572/lib/python3.11/site-packages/inspect_evals/ifeval/ifeval.py

## Training Data Task-Targetted Cleaning

### MetaMathQA

In [6]:
# MetaMathQA

def is_numeric_answer(response):
    # Extract what comes after "The answer is:"
    match = re.search(r'The answer is:\s*(.+)$', response.strip())
    if not match:
        return False
    ans = match.group(1).strip()
    # Remove \boxed{} wrapper if present
    ans = re.sub(r'\\boxed\{([^}]+)\}', r'\1', ans)
    try:
        float(ans.replace(',', ''))
        return True
    except:
        return False

numeric = sum(1 for ex in math_train if is_numeric_answer(ex['response']))
print(f"Numeric answers: {numeric} / {len(math_train)} ({100*numeric/len(math_train):.1f}%)")

# Also check type distribution
# types = Counter(ex['type'] for ex in math_train)
# print(types)

Numeric answers: 369767 / 395000 (93.6%)
Counter({'GSM_Rephrased': 80000, 'GSM_AnsAug': 80000, 'MATH_AnsAug': 75000, 'MATH_Rephrased': 50000, 'GSM_SV': 40000, 'GSM_FOBAR': 40000, 'MATH_FOBAR': 15000, 'MATH_SV': 15000})


In [7]:
# Filter to GSM8K-origin only — these have numeric answers
gsm8k_origin_types = [t for t in types if 'GSM' in t or 'gsm' in t]
print("GSM8K-origin types:", gsm8k_origin_types)

math_filtered = math_train.filter(
    lambda ex: any(t in ex['type'] for t in ['GSM', 'gsm'])
)
print("Filtered size:", len(math_filtered))

GSM8K-origin types: ['GSM_Rephrased', 'GSM_SV', 'GSM_FOBAR', 'GSM_AnsAug']
Filtered size: 240000


In [36]:
print("=== MetaMathQA (filtered on output format) ===")
print("Columns:", math_filtered.column_names)
print("Example:")
print("  query:",math_filtered[0]['query'])
print("  answer:", math_filtered[0]['response'])

=== MetaMathQA (filtered on output format) ===
Columns: ['type', 'query', 'original_question', 'response']
Example:
  query: What is the total cost of purchasing equipment for all sixteen players on the football team, considering that each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80?
  answer: Each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80.
So the total cost for each player is $25 + $15.20 + $6.80 = $47.
Since there are sixteen players on the football team, the total cost for all of them is 16 * $47 = $752.
#### 752
The answer is: 752


In [8]:
MATH_PROMPT_TEMPLATE = """Solve the following math problem step by step. The last line of your response should be of the form "ANSWER: $ANSWER" (without quotes) where $ANSWER is the answer to the problem.

{prompt}

Remember to put your answer on its own line at the end in the form "ANSWER: $ANSWER" (without quotes) where $ANSWER is the answer to the problem, and you do not need to use a \\boxed command.

Reasoning:""".strip()

def format_math_example(ex):
    # Convert "The answer is: 18" → "ANSWER: 18"
    response = ex['response']
    response = re.sub(r'The answer is:\s*(.+)$', r'ANSWER: \1', response.strip())
    return {
        "messages": [
            {"role": "user",    "content": MATH_PROMPT_TEMPLATE.format(prompt=ex['query'])},
            {"role": "assistant","content": response}
        ]
    }

In [9]:
# Apply the formatting to MetaMathQA (GSM-origin only)
math_formatted = [format_math_example(ex) for ex in math_filtered]
print(f"Formatted {len(math_formatted)} math examples")
print("\nSample:")
print(math_formatted[0]['messages'][0]['content'])
print("---")
print(math_formatted[0]['messages'][1]['content'])

Formatted 240000 math examples

Sample:
Solve the following math problem step by step. The last line of your response should be of the form "ANSWER: $ANSWER" (without quotes) where $ANSWER is the answer to the problem.

What is the total cost of purchasing equipment for all sixteen players on the football team, considering that each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80?

Remember to put your answer on its own line at the end in the form "ANSWER: $ANSWER" (without quotes) where $ANSWER is the answer to the problem, and you do not need to use a \boxed command.

Reasoning:
---
Each player requires a $25 jersey, a $15.20 pair of shorts, and a pair of socks priced at $6.80.
So the total cost for each player is $25 + $15.20 + $6.80 = $47.
Since there are sixteen players on the football team, the total cost for all of them is 16 * $47 = $752.
#### 752
ANSWER: 752


### GSM8K training set

In [10]:
# GSM8K training set
def format_gsm8k_example(ex: dict) -> dict:
    parts = ex["answer"].split("####")
    reasoning = parts[0].strip()
    answer = parts[1].strip()
    return {
        "messages": [
            {
                "role": "user",
                "content": MATH_PROMPT_TEMPLATE.format(prompt=ex["question"])
            },
            {
                "role": "assistant",
                "content": f"{reasoning}\nANSWER: {answer}"
            }
        ]
    }

gsm8k_formatted = [format_gsm8k_example(ex) for ex in gsm8k_train]

In [11]:
# Combine GSM8K train + MetaMathQA GSM-origin
math_all = gsm8k_formatted + math_formatted
print(f"Total math training examples: {len(math_all)}")

Total math training examples: 247473


### Tulu-3 (IFEVal)

- CoCoNot (ODC-BY-1.0), 10,983 prompts (Brahman et al., 2024)
- FLAN v2 via ai2-adapt-dev/flan_v2_converted, 89,982 prompts (Longpre et al., 2023)
- No Robots (CC-BY-NC-4.0), 9,500 prompts (Rajani et al. 2023) ------ human-written instructions, many with format constraints. maybe useful for IFEval
- OpenAssistant Guanaco (Apache 2.0), 7,132 prompts (Kopf et al., 2024)
- Tulu 3 Persona MATH (ODC-BY-1.0), 149,960 prompts ------ not purely numeric (x) 
- Tulu 3 Persona GSM (ODC-BY-1.0), 49,980 prompts ------ GSM8K (v)
- Tulu 3 Persona Python (ODC-BY-1.0), 34,999 prompts ------ HumanEval (v)
- Tulu 3 Persona Algebra (ODC-BY-1.0), 20,000 prompts ------ not purely numeric (x)
- $\color{orange}{Tulu 3 Persona IF (ODC-BY-1.0), 29,980 prompts}$  ------ IFEval (v)
- NuminaMath-TIR (Apache 2.0), 64,312 prompts (Beeching et al. 2024)
- Tulu 3 WildGuardMix (Apache 2.0), 50,000 prompts (Han et al., 2024)
- Tulu 3 WildJailbreak (ODC-BY-1.0), 50,000 prompts (Wildteaming, 2024)
- Tulu 3 Hardcoded (CC-BY-4.0), 240 prompts
- Aya (Apache 2.0), 100,000 prompts (Singh et al., 2024)
- WildChat GPT-4 (ODC-BY-1.0), 100,000 prompts (Zhao et al., 2024)
- TableGPT (MIT), 5,000 prompts (Zha et al., 2023)
- SciRIFF (ODC-BY-1.0), 10,000 prompts (Wadden et al., 2024)
- Evol CodeAlpaca (Apache 2.0), 107,276 prompts (Luo et al., 2023)

In [15]:
IFEVAL_CONSTRAINT_KEYWORDS = [
    # Keywords group
    "include the word", "include the phrase", "include keyword",
    "the word", "should appear", "times", "do not include", "forbidden",
    "refrain from using", "the letter",
    # Language
    "entire response should be in", "no other language",
    # Length constraints
    "number of paragraphs", "number of words", "number of sentences",
    "at least", "at most", "around", "first word",
    # Detectable content
    "postscript", "p.s.", "placeholder", "square bracket",
    # Detectable format
    "bullet point", "bullet points", "title", "double angular bracket",
    "<<", ">>", "choose one of", "highlighted section", "highlight",
    "sections", "section splitter", "json format", "json",
    "repeat the request", "two responses", "******",
    # Change cases
    "all capital", "all uppercase", "all lowercase",
    "capital letters only", "no capital",
    # Start/end
    "finish your response with", "end your response with",
    "wrap your entire", "double quotation", "start with", "end with",
    # Punctuation
    "no comma", "no commas", "refrain from",
]

In [16]:
SOURCE_CONFIG = {
    # Use all — purpose-built for IFEval
    'ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980': {
        'filter': False,
        'cap': None,
    },
    # Use all — small, high quality human-written
    'ai2-adapt-dev/no_robots_converted': {
        'filter': False,
        'cap': None,
    },
    # Filter for constraint-rich examples, cap to avoid dominating
    'ai2-adapt-dev/flan_v2_converted': {
        'filter': True,
        'cap': 30_000,
    },
}

In [18]:
def is_constraint_rich(ex: dict) -> bool:
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    ).lower()
    return any(kw in user_turn for kw in IFEVAL_CONSTRAINT_KEYWORDS)

In [19]:
def collect_if_data_safe() -> list:
    if_train = load_dataset(
        "allenai/tulu-3-sft-mixture", split="train", streaming=True
    )

    buckets = defaultdict(list)
    
    for ex in if_train:
        src = ex['source']
        if src not in SOURCE_CONFIG:
            continue
        
        config = SOURCE_CONFIG[src]
        cap = config['cap']
        
        # Skip if this source is already full
        if cap and len(buckets[src]) >= cap:
            continue
        
        # Apply keyword filter if configured
        if config['filter'] and not is_constraint_rich(ex):
            continue
        
        buckets[src].append({
            "messages": ex['messages'],
            "source": src
        })
    
    print("IF data collected:")
    total = 0
    for src, examples in buckets.items():
        print(f"  {src.split('/')[-1]}: {len(examples)}")
        total += len(examples)
    print(f"  Total: {total}")
    
    return [ex for examples in buckets.values() for ex in examples]

if_formatted = collect_if_data_safe()

IF data collected:
  flan_v2_converted: 20234
  no_robots_converted: 9500
  personahub_ifdata_manual_seed_v3_29980: 29980
  Total: 59714


In [ ]:
if_train = load_dataset("allenai/tulu-3-sft-mixture", split="train")  # no streaming=True
if_filtered = if_train.filter(lambda ex: ex['source'] == 'ai2-adapt-dev/personahub_ifdata_manual_seed_v3_29980')
print(len(if_filtered))

Filter: 100%|██████████| 939343/939343 [00:19<00:00, 47438.10 examples/s]

29980


### CodeAlpaca / OpenCodeInstruct (HumanEval)

In [40]:
# ── CodeAlpaca ─────────────────────────────────────────────────────

HUMANEVAL_INSTRUCTION = """Read the following function signature and docstring, and fully implement the function described. Your response should only contain the code for this function.\n"""

# Load as regular dataset
code_alpaca_train = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

code_alpaca_formatted = code_alpaca_train.map(
    lambda ex: {
        "messages": [
            {"role": "user",      "content": HUMANEVAL_INSTRUCTION + ex['input']},
            {"role": "assistant", "content": ex['output']}
        ]
    },
    num_proc=4,
    remove_columns=code_alpaca_train.column_names  # drop unused columns
)

# print(f"Code examples (score=1.0): {len(code_formatted)}")
print(f"Code examples: {len(code_alpaca_formatted)}")

Code examples: 20022


In [41]:
# ── OpenCodeInstruct ─────────────────────────────────────────────────────

HUMANEVAL_INSTRUCTION = """Read the following function signature and docstring, and fully implement the function described. Your response should only contain the code for this function.\n"""

# Load as regular dataset
code_open_train = load_dataset("nvidia/OpenCodeInstruct", split="train", num_proc=4)

# Filter first (fast, vectorized), then format
code_open_filtered = code_open_train.filter(
    lambda ex: float(ex['average_test_score']) >= 1.0,
    num_proc=4
)

code_open_formatted = code_open_filtered.map(
    lambda ex: {
        "messages": [
            {"role": "user",      "content": HUMANEVAL_INSTRUCTION + ex['input']},
            {"role": "assistant", "content": ex['output']}
        ]
    },
    num_proc=4,
    remove_columns=code_open_filtered.column_names  # drop unused columns
)

# print(f"Code examples (score=1.0): {len(code_formatted)}")
print(f"Code examples: {len(code_open_formatted)}")

Code examples: 1641305


## Data selection and filtering

At this point, we have `math_all` (247,473), `if_formatted` (broader, coarse, 59,714), `if_filtered` (smaller, fine, 29,980), and `code_formatted` (20,022).

### Deduplication

In [ ]:
# !pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 18.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [datasketch]


In [21]:
from datasketch import MinHash, MinHashLSH

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def get_shingles(text: str, k: int = 5) -> set:
    tokens = normalize(text).split()
    return set(' '.join(tokens[i:i+k]) for i in range(len(tokens) - k + 1))

def get_user_turn(ex: dict) -> str:
    return next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )

def deduplicate(examples: list, threshold: float = 0.85) -> list:
    """
    Remove near-duplicates using MinHash LSH.
    threshold: Jaccard similarity above which two examples are considered duplicates.
    """
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    unique = []
    
    for i, ex in enumerate(examples):
        text = get_user_turn(ex)
        shingles = get_shingles(text)
        if not shingles:
            continue
        
        m = MinHash(num_perm=128)
        for s in shingles:
            m.update(s.encode('utf8'))
        
        key = str(i)
        try:
            result = lsh.query(m)
            if not result:  # no near-duplicates found
                lsh.insert(key, m)
                unique.append(ex)
        except Exception:
            lsh.insert(key, m)
            unique.append(ex)
    
    print(f"  Deduplication: {len(examples)} → {len(unique)} ({len(examples)-len(unique)} removed)")
    return unique

### Quality filtering

In [22]:
import json

# ── Math ─────────────────────────────────────────────────────

def quality_filter_math(ex: dict) -> bool:
    assistant_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'assistant'), ""
    )
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )
    
    # Must have ANSWER: at the end (matches eval format)
    if 'ANSWER:' not in assistant_turn:
        return False
    
    # Extract and validate answer is numeric
    match = re.search(r'ANSWER:\s*(.+)$', assistant_turn.strip())
    if not match:
        return False
    answer_str = match.group(1).strip().replace(',', '')
    try:
        float(answer_str)
    except ValueError:
        return False
    
    # Must have reasoning (not just the answer)
    lines = [l.strip() for l in assistant_turn.strip().split('\n') if l.strip()]
    if len(lines) < 2:
        return False
    
    # Reject nonsensically short responses
    if len(assistant_turn.split()) < 20:
        return False
    
    return True


# ── Code ─────────────────────────────────────────────────────

def quality_filter_code(ex: dict) -> bool:
    assistant_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'assistant'), ""
    )
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )
    
    # Must contain actual code
    has_code = '```' in assistant_turn or 'def ' in assistant_turn
    if not has_code:
        return False
    
    # Must be Python (HumanEval is Python-only)
    is_python = (
        '```python' in assistant_turn.lower() or
        'def ' in assistant_turn or
        'import ' in assistant_turn
    )
    if not is_python:
        return False
    
    # Reject extremely short solutions (likely incomplete)
    code_content = re.sub(r'```(?:python)?\n?(.*?)```', r'\1', assistant_turn, flags=re.DOTALL)
    if len(code_content.strip().split('\n')) < 2:
        return False
    
    return True


# ── IF ───────────────────────────────────────────────────────

def quality_filter_if(ex: dict) -> bool:
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    )
    assistant_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'assistant'), ""
    )
    
    # Must have non-trivial response
    if len(assistant_turn.split()) < 10:
        return False
    
    # Reject refusals — these appear in tulu safety sources that might slip through
    refusal_phrases = [
        "i cannot", "i can't", "i'm unable", "i am unable",
        "i won't", "i will not", "as an ai", "as a language model"
    ]
    if any(p in assistant_turn.lower() for p in refusal_phrases):
        return False
    
    # Verify response actually attempts to satisfy constraints
    # Check for no_comma constraint
    if 'no comma' in user_turn.lower() or 'no commas' in user_turn.lower():
        if ',' in assistant_turn:
            return False
    
    # Check for all caps constraint
    if 'all capital' in user_turn.lower() or 'capital letters only' in user_turn.lower():
        if assistant_turn != assistant_turn.upper():
            return False
    
    # Check for JSON format constraint
    if 'json format' in user_turn.lower():
        try:
            json.loads(assistant_turn)
        except json.JSONDecodeError:
            return False
    
    return True

### Difficulty-based selection

In [ ]:
# ── Math difficulty ───────────────────────────────────────────

def math_difficulty_score(ex: dict) -> float:
    """Proxy: number of reasoning steps (lines with calculations)."""
    assistant_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'assistant'), ""
    )
    # Count lines with arithmetic operations
    calc_lines = sum(
        1 for line in assistant_turn.split('\n')
        if any(op in line for op in ['=', '+', '-', '*', '/'])
    )
    return calc_lines


# ── Code difficulty ───────────────────────────────────────────

def code_difficulty_score(ex: dict) -> float:
    """Proxy: number of logical branches (if/for/while/try)."""
    assistant_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'assistant'), ""
    )
    keywords = ['if ', 'elif ', 'for ', 'while ', 'try:', 'except', 'with ']
    return sum(assistant_turn.count(kw) for kw in keywords)


# ── IF difficulty ─────────────────────────────────────────────

def if_difficulty_score(ex: dict) -> float:
    """Proxy: number of constraint keywords in the instruction."""
    user_turn = next(
        (m['content'] for m in ex['messages'] if m['role'] == 'user'), ""
    ).lower()
    return sum(1 for kw in IFEVAL_CONSTRAINT_KEYWORDS if kw in user_turn)


# ── Difficulty-stratified sampling ───────────────────────────

def difficulty_stratified_sample(examples: list, score_fn, n: int, p_easy=0.20, p_medium=0.50, p_hard=0.30) -> list:
    """
    Sample n examples stratified across easy/medium/hard terciles.
    Avoids over-representing trivially easy or impossibly hard examples.
    """
    scored = [(score_fn(ex), ex) for ex in examples]
    scored.sort(key=lambda x: x[0])
    
    tercile = len(scored) // 3
    easy   = [ex for _, ex in scored[:tercile]]
    medium = [ex for _, ex in scored[tercile:2*tercile]]
    hard   = [ex for _, ex in scored[2*tercile:]]
    
    # Sample proportionally: 20% easy, 50% medium, 30% hard
    # Harder examples give more learning signal
    n_easy   = int(n * p_easy)
    n_medium = int(n * p_medium)
    n_hard   = int(n * p_hard)
    
    sampled = (
        random.sample(easy,   min(n_easy,   len(easy)))   +
        random.sample(medium, min(n_medium, len(medium))) +
        random.sample(hard,   min(n_hard,   len(hard)))
    )
    
    print(f"  Difficulty split: {n_easy} easy / {n_medium} medium / {n_hard} hard")
    return sampled

### Full Pipeline

In [ ]:
import random

def process_task(name: str, examples: list, quality_fn, difficulty_fn, target_n: int) -> list:
    print(f"\n=== {name} ===")
    print(f"  Input: {len(examples)}")
    
    # Step 1: quality filter
    examples = [ex for ex in examples if quality_fn(ex)]
    print(f"  After quality filter: {len(examples)}")
    
    # Step 2: deduplicate
    examples = deduplicate(examples, threshold=0.85)
    print(f"  After deduplication: {len(examples)}")
    
    # Step 3: difficulty-stratified sample
    if len(examples) > target_n:
        examples = difficulty_stratified_sample(examples, difficulty_fn, target_n)
    print(f"  Final: {len(examples)}")
    
    return examples

TARGET_N = 10_000

math_final = process_task(
    "Math", math_all,
    quality_filter_math, math_difficulty_score,
    TARGET_N
)

code_alpaca_final = process_task(
    "Code", code_alpaca_formatted,
    quality_filter_code, code_difficulty_score,
    TARGET_N
)

code_open_final = process_task(
    "Code", code_open_formatted,
    quality_filter_code, code_difficulty_score,
    TARGET_N - len(code_alpaca_final)
)

if_final = process_task(
    "IF", if_filtered, # if_formatted
    quality_filter_if, if_difficulty_score,
    TARGET_N  # or len(if_formatted) to keep all
)


=== Math ===
  Input: 247473
  After quality filter: 246997
  Deduplication: 246997 → 66334 (180663 removed)
  After deduplication: 66334
  Difficulty split: 9000 easy / 22500 medium / 13500 hard
  Final: 44611

=== Code ===
  Input: 1641305
  After quality filter: 1641155
  Deduplication: 1641155 → 1460282 (180873 removed)
  After deduplication: 1460282
  Difficulty split: 9000 easy / 22500 medium / 13500 hard
  Final: 45000

=== IF ===
  Input: 29980
  After quality filter: 24766
  Deduplication: 24766 → 24764 (2 removed)
  After deduplication: 24764
  Final: 24764

Final training set: 114375


In [ ]:
code_alpaca_final = [dict(ex) for ex in code_alpaca_final]
code_open_final = [dict(ex) for ex in code_open_final]
code_final = code_alpaca_final + code_open_final
random.shuffle(code_final)
print(f"\nFinal training set for code generation: {len(code_final)}")

## Save the data

In [ ]:
# Tag each example with task_id before combining
math_tagged = [{**ex, "task_id": 0} for ex in math_final]
code_tagged  = [{**ex, "task_id": 2} for ex in code_final]
if_tagged    = [{**ex, "task_id": 1} for ex in if_final]

training_data = math_tagged + code_tagged + if_tagged
random.shuffle(training_data)
print(f"\nFinal training set: {len(training_data)}")


Final training set: 29999


In [ ]:
# remove near-duplicate in test data
training_data = [ 
        ex for i, ex in enumerate(training_data)
        if i != 12452
    ]

In [ ]:
import json
with open("training_data.jsonl", "w") as f:
    for ex in training_data:
        f.write(json.dumps(ex) + "\n")

print(f"Saved {len(training_data)} examples")

Saved 29999 examples
